In [1]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
BASE_PATH = os.path.expanduser("~/100xDevs/personality/Personality-Classifier")

# Raw CelebA images
IMAGE_SOURCE = os.path.join(
    BASE_PATH,
    "data/raw/ZipCeleb/img_align_celeba/img_align_celeba"
)

# CelebA attributes file
ANNOTATION_FILE = os.path.join(
    BASE_PATH,
    "data/raw/ZipCeleb/list_attr_celeba.csv"
)

# Final processed dataset output
OUTPUT_BASE = os.path.join(
    BASE_PATH,
    "data/processed/double_chin_dataset"
)

print("Image Source:", IMAGE_SOURCE)
print("Annotation File:", ANNOTATION_FILE)
print("Output Base:", OUTPUT_BASE)

Image Source: /home/yeezy/100xDevs/personality/Personality-Classifier/data/raw/ZipCeleb/img_align_celeba/img_align_celeba
Annotation File: /home/yeezy/100xDevs/personality/Personality-Classifier/data/raw/ZipCeleb/list_attr_celeba.csv
Output Base: /home/yeezy/100xDevs/personality/Personality-Classifier/data/processed/double_chin_dataset


In [3]:
# Load CelebA CSV
data = pd.read_csv(ANNOTATION_FILE)

print("Total rows:", len(data))
print("Columns:", data.columns.tolist())

# Extract only required columns
df = data[["image_id", "Double_Chin"]].copy()

# Rename for clarity
df.rename(columns={"image_id": "Image_Name"}, inplace=True)

# Convert -1 to 0
df["Double_Chin"] = df["Double_Chin"].replace(-1, 0).astype(int)

print("\nClass Distribution:")
print(df["Double_Chin"].value_counts())

Total rows: 202599
Columns: ['image_id', '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young']

Class Distribution:
Double_Chin
0    193140
1      9459
Name: count, dtype: int64


In [4]:
train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["Double_Chin"],
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(valid_df))

Train size: 162079
Validation size: 40520


In [5]:
for split in ["train", "valid"]:
    for label in ["double_chin", "no_double_chin"]:
        os.makedirs(
            os.path.join(OUTPUT_BASE, split, label),
            exist_ok=True
        )

print("Folder structure created.")

Folder structure created.


In [6]:
def copy_images(dataframe, split_name):
    copied = 0
    skipped = 0

    for _, row in dataframe.iterrows():
        image_name = row["Image_Name"]
        label = "double_chin" if row["Double_Chin"] == 1 else "no_double_chin"

        source_path = os.path.join(IMAGE_SOURCE, image_name)
        destination_path = os.path.join(
            OUTPUT_BASE,
            split_name,
            label,
            image_name
        )

        if os.path.exists(source_path):
            shutil.copy(source_path, destination_path)
            copied += 1
        else:
            skipped += 1

    print(f"{split_name.upper()} -> Copied: {copied}, Skipped: {skipped}")

In [7]:
copy_images(train_df, "train")
copy_images(valid_df, "valid")

print("\n✅ Double Chin dataset creation complete.")

TRAIN -> Copied: 162079, Skipped: 0
VALID -> Copied: 40520, Skipped: 0

✅ Double Chin dataset creation complete.
